# Stage 3: Streaming Health Check
## ISM 6562 — Final Project — MediStream Telehealth


## Setup

In [ ]:
!pip install -q kafka-python==2.0.2

In [ ]:
from pyspark.sql import SparkSession, functions as F
from kafka import KafkaAdminClient, KafkaConsumer, TopicPartition

KAFKA_BOOTSTRAP = 'kafka:9093'
TOPIC_METRICS = 'session-metrics'
TOPIC_ALERTS  = 'session-alerts'

spark = (SparkSession.builder
    .appName('MediStream-Stage3c-HealthCheck')
    .master('spark://spark-master:7077')
    .config('spark.executor.memory', '512m')
    .config('spark.driver.memory', '512m')
    .getOrCreate())

## Check 1 + 2: Topics exist with expected configuration

In [ ]:
EXPECTED_TOPICS = {
    TOPIC_METRICS: 4,  # partitions per kafka-init/create-topics.sh
    TOPIC_ALERTS:  2,
}

admin = KafkaAdminClient(bootstrap_servers=[KAFKA_BOOTSTRAP])
live_topics = admin.list_topics()
print(f'Topics on broker: {sorted(live_topics)}')

missing = [t for t in EXPECTED_TOPICS if t not in live_topics]
if missing:
    print(f'  FAIL — missing topics: {missing}')
    print('  Fix: docker exec -it medistream-kafka bash /kafka-init/create-topics.sh')
else:
    md = admin.describe_topics(list(EXPECTED_TOPICS.keys()))
    for entry in md:
        name = entry['topic']
        actual_parts = len(entry['partitions'])
        expected = EXPECTED_TOPICS[name]
        status = 'OK' if actual_parts == expected else f'WARN (expected {expected})'
        print(f'  {name:<20s} partitions={actual_parts}  {status}')

## Check 3: Topic offsets — proof that data flowed

In [ ]:
def end_offsets(topic):
    consumer = KafkaConsumer(bootstrap_servers=[KAFKA_BOOTSTRAP],
                             enable_auto_commit=False)
    parts = consumer.partitions_for_topic(topic)
    if not parts:
        consumer.close()
        return None
    tps = [TopicPartition(topic, p) for p in parts]
    end = consumer.end_offsets(tps)
    consumer.close()
    return {tp.partition: off for tp, off in end.items()}

for topic in EXPECTED_TOPICS:
    offsets = end_offsets(topic)
    if offsets is None:
        print(f'  {topic:<20s} not found')
        continue
    total = sum(offsets.values())
    status = 'OK' if total > 0 else 'WARN — no data'
    print(f'  {topic:<20s} total_offset={total:>8,}  per-partition={offsets}  {status}')

## Check 4: HDFS streaming_alerts table has rows + expected schema

In [ ]:
EXPECTED_ALERT_COLS = {
    'window_start', 'window_end', 'session_id', 'appointment_id',
    'device_type', 'os', 'sample_count', 'avg_latency_ms',
    'avg_packet_loss_pct', 'avg_audio_quality_score', 'alert_type',
    'physician_id', 'specialty', 'visit_type', 'suggested_action', 'alert_date',
}

HDFS_ALERTS = 'hdfs://namenode:9000/medistream/analytics/streaming_alerts'
LOCAL_ALERTS = '/home/jovyan/data/analytics/streaming_alerts'

alerts_df = None
for path in (HDFS_ALERTS, LOCAL_ALERTS):
    try:
        alerts_df = spark.read.parquet(path)
        print(f'Loaded alerts from {path}')
        break
    except Exception as e:
        print(f'  miss {path}: {type(e).__name__}')

if alerts_df is None:
    print('  FAIL — no streaming_alerts found in HDFS or local mount')
else:
    cols = set(alerts_df.columns)
    missing = EXPECTED_ALERT_COLS - cols
    extra = cols - EXPECTED_ALERT_COLS
    n = alerts_df.count()
    print(f'  rows: {n:,}')
    print(f'  cols: {sorted(cols)}')
    if missing:
        print(f'  WARN — missing columns: {sorted(missing)}')
    if extra:
        print(f'  INFO — extra columns: {sorted(extra)}')
    if n > 0 and not missing:
        print('  OK')

## Check 5: Partition layout

In [ ]:
if alerts_df is not None:
    sc = spark.sparkContext
    fs = sc._jvm.org.apache.hadoop.fs.FileSystem.get(sc._jsc.hadoopConfiguration())
    base_path = HDFS_ALERTS if 'hdfs' in str(alerts_df.inputFiles()[:1]) else LOCAL_ALERTS
    try:
        path = sc._jvm.org.apache.hadoop.fs.Path(base_path)
        statuses = fs.listStatus(path)
        date_parts = [s.getPath().getName() for s in statuses if s.isDirectory() and s.getPath().getName().startswith('alert_date=')]
        print(f'  {len(date_parts)} alert_date partitions:')
        for p in sorted(date_parts):
            print(f'    {p}')
    except Exception as e:
        print(f'  layout check skipped: {e}')

## Check 6: Sample of recent alerts (visual sanity)

In [ ]:
if alerts_df is not None and alerts_df.count() > 0:
    print('Most recent 15 alert windows:')
    (alerts_df
        .orderBy(F.desc('window_end'))
        .select('window_end', 'session_id', 'specialty', 'alert_type',
                'avg_latency_ms', 'avg_packet_loss_pct', 'avg_audio_quality_score',
                'device_type', 'os', 'suggested_action')
        .show(15, truncate=False))

    print('\nAlert type distribution:')
    alerts_df.groupBy('alert_type').count().orderBy(F.desc('count')).show()

    print('\nAlerts per device_type:')
    alerts_df.groupBy('device_type').count().orderBy(F.desc('count')).show()